In [3]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import (
    RandomForestClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
    StackingClassifier
)
from xgboost import XGBClassifier

In [4]:
# processed data loading
import pandas as pd

train_df = pd.read_csv("../data/processed/train.csv")
test_df = pd.read_csv("../data/processed/test.csv")

In [6]:
# spllitting into X and y

X_train = train_df.drop('HeartDisease', axis = 1)
y_train = train_df['HeartDisease']

X_test = test_df.drop('HeartDisease', axis = 1)
y_test = test_df['HeartDisease']

In [15]:
# models dictionary

models = {
    "Logistic Regression" : LogisticRegression(),
    "KNN" : KNeighborsClassifier(),
    "Naive Bayes" : GaussianNB(),
    "Decision Tree" : DecisionTreeClassifier(),
    "Random Forest" : RandomForestClassifier(),
    "SVM" : SVC(probability= True),            # SVC don't return probability by default
    "AdaBoost" : AdaBoostClassifier(),
    "Gradient Boost" : GradientBoostingClassifier(),
    "XGBoost" : XGBClassifier()
}

In [16]:
# training models

trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    trained_models[name] = model
    
print("All models trained successfully")

All models trained successfully


In [17]:
# Initial model evaluation 

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,          # receiver operating characteristic area under curve
    classification_report,
    confusion_matrix
)

In [18]:
result = []

for name, model in trained_models.items():
    
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:,1]
    
    result.append({
        "Model" : name,
        "Accuracy" : accuracy_score(y_test, y_pred),
        "Precision" : precision_score(y_test, y_pred),
        "Recall" : recall_score(y_test, y_pred),
        "F1 Score" : f1_score(y_test, y_pred),
        "ROC-AUC" : roc_auc_score(y_test, y_prob)
    })
    
result_df = pd.DataFrame(result).sort_values(
    by = "F1 Score",
    ascending= False
)

result_df

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
6,AdaBoost,0.896739,0.910891,0.901961,0.906404,0.924079
4,Random Forest,0.880435,0.892157,0.892157,0.892157,0.930536
0,Logistic Regression,0.875000,0.883495,0.892157,0.887805,0.933286
7,Gradient Boost,0.869565,0.882353,0.882353,0.882353,0.909971
2,Naive Bayes,0.869565,0.906250,0.852941,0.878788,0.944285
5,SVM,0.864130,0.881188,0.872549,0.876847,0.934601
1,KNN,0.853261,0.864078,0.872549,0.868293,0.928623
8,XGBoost,0.831522,0.858586,0.833333,0.845771,0.911645
3,Decision Tree,0.733696,0.811765,0.676471,0.737968,0.740674


In [20]:
# selecting best 6 models based on f1_score for hyperparameter tuning

models_best6 = result_df.head(6)
models_best6

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
6,AdaBoost,0.896739,0.910891,0.901961,0.906404,0.924079
4,Random Forest,0.880435,0.892157,0.892157,0.892157,0.930536
0,Logistic Regression,0.875000,0.883495,0.892157,0.887805,0.933286
7,Gradient Boost,0.869565,0.882353,0.882353,0.882353,0.909971
2,Naive Bayes,0.869565,0.906250,0.852941,0.878788,0.944285
5,SVM,0.864130,0.881188,0.872549,0.876847,0.934601


In [21]:
# Hyperparameter tuning

from sklearn.model_selection import(
    GridSearchCV,
    RandomizedSearchCV,
    cross_val_score
)

In [23]:
# function for model tuning

def tune_model(model, params, X_train, y_train):
    rs = RandomizedSearchCV(
        estimator= model,
        param_distributions= params,
        n_iter= 20,
        scoring= 'f1',
        cv= 5,
        verbose= 1,
        n_jobs= -1,  # controls CPU usage, -1 means use all the available CPU cores to make the search faster
        random_state= 42
    )
    
    rs.fit(X_train, y_train)
    return rs.best_estimator_, rs.best_params_, rs.best_score_

In [27]:
# AdaBoost

best_ada, ada_params, ada_score = tune_model(
    AdaBoostClassifier(),
    {
        "n_estimators" : [50, 100, 200],
        "learning_rate" : [0.01, 0.1, 1]
    },
    X_train,
    y_train
)

print("Best Parameter : ",ada_params)
print("Best F1 : ",ada_score)

c:\Users\kashy\Projects\Heart-disease-prediction\.venv\Lib\site-packages\sklearn\model_selection\_search.py:317: UserWarning: The total space of parameters 9 is smaller than n_iter=20. Running 9 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 5 folds for each of 9 candidates, totalling 45 fits
Best Parameter :  {'n_estimators': 100, 'learning_rate': 1}
Best F1 :  0.8737335651100221


In [31]:
# Random Forest

best_rf, rf_params, rf_score = tune_model(
    RandomForestClassifier(),
    {
        "n_estimators" : [100, 200, 300, 500],
        "max_depth" : [3, 5, 10, None],
        "min_samples_split" : [2, 5, 10]
    },
    X_train,
    y_train
)

print("Best Parameters: ",rf_params)
print("Best F1: ",rf_score)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best Parameters:  {'n_estimators': 500, 'min_samples_split': 5, 'max_depth': None}
Best F1:  0.8860285324847854


In [32]:
# Logistic regression

best_lr, lr_params, lr_score = tune_model(
    LogisticRegression(),
    {
        "C": [0.01, 0.1, 1, 10, 100],
        "solver": ['liblinear', 'lbfgs']
    },
    X_train,
    y_train 
)

print("Best parameters: ",lr_params)
print("Best F1: ",lr_score)

c:\Users\kashy\Projects\Heart-disease-prediction\.venv\Lib\site-packages\sklearn\model_selection\_search.py:317: UserWarning: The total space of parameters 10 is smaller than n_iter=20. Running 10 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best parameters:  {'solver': 'liblinear', 'C': 0.1}
Best F1:  0.8688566039635474


In [33]:
# gradient boost

best_gb, gb_params, gb_score = tune_model(
    GradientBoostingClassifier(),
    {
        "n_estimators" : [100, 200, 300],
        "learning_rate" : [0.01, 0.05, 0.1],
        "max_depth" : [3, 5, 7]
    },
    X_train,
    y_train
)

print("Best Parameters : ",gb_params)
print("Best F1: ",gb_score)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best Parameters :  {'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.05}
Best F1:  0.8707851306246404


In [34]:
# naive bayes  

best_nb, nb_params, nb_score = tune_model(
    GaussianNB(),
    {
        "var_smoothing": [1e-9, 1e-8, 1e-7]
    },
    X_train,
    y_train 
)
print("Best Parameters:", nb_params)
print("Best F1:", nb_score)

Fitting 5 folds for each of 3 candidates, totalling 15 fits
Best Parameters: {'var_smoothing': 1e-09}
Best F1: 0.8544876931373713


c:\Users\kashy\Projects\Heart-disease-prediction\.venv\Lib\site-packages\sklearn\model_selection\_search.py:317: UserWarning: The total space of parameters 3 is smaller than n_iter=20. Running 3 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


In [35]:
# SVM

best_svm, svm_params, svm_score = tune_model(
    SVC(probability= True),
    {
        "C": [0.1, 1, 10, 100],
        "kernel": ["linear", "rbf"],
        "gamma": ['scale', 'auto']
    },
    X_train,
    y_train
)

print("Best Parameters:", svm_params)
print("Best F1:", svm_score)

c:\Users\kashy\Projects\Heart-disease-prediction\.venv\Lib\site-packages\sklearn\model_selection\_search.py:317: UserWarning: The total space of parameters 16 is smaller than n_iter=20. Running 16 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Fitting 5 folds for each of 16 candidates, totalling 80 fits
Best Parameters: {'kernel': 'rbf', 'gamma': 'scale', 'C': 1}
Best F1: 0.875189974602159


In [37]:
# Comparing the F1 score of all models to extract best 3 models

results = {
    "Logistic Regression": lr_score,
    "Random Forest": rf_score,
    "Gradient Boost": gb_score,
    "Ada Boost": ada_score,
    "SVM": svm_score,
    "Naive Bayes": nb_score
    
}

result_df = pd.DataFrame(
    list(results.items()),
    columns= ["Models", "F1 Score"]
).sort_values(
    by= "F1 Score",
    ascending= False
)

result_df

,Models,F1 Score
1,Random Forest,0.886029
4,SVM,0.875190
3,Ada Boost,0.873734
2,Gradient Boost,0.870785
0,Logistic Regression,0.868857
5,Naive Bayes,0.854488


In [39]:
models_best3 = result_df.head(3)
models_best3

,Models,F1 Score
1,Random Forest,0.886029
4,SVM,0.875190
3,Ada Boost,0.873734


In [40]:
# Applying stacking classifier on best 3 models

estimators = [
    ("rf", best_rf),
    ("svm", best_svm),
    ("ada", best_ada)
]

In [46]:
stack_model = StackingClassifier(
    estimators= estimators,
    final_estimator= LogisticRegression(),
    cv = 5,
    n_jobs= -1
)

stack_model.fit(X_train, y_train)

,estimators,"[('rf', ...), ('svm', ...), ...]"
,final_estimator,LogisticRegression()
,cv,5
,stack_method,'auto'
,n_jobs,-1
,passthrough,False
,verbose,0
,n_estimators,500
,criterion,'gini'
,max_depth,None
,min_samples_split,5


In [47]:
y_pred = stack_model.predict(X_test)
y_prob = stack_model.predict_proba(X_test)[:,1]

In [48]:
stacking_f1 = f1_score(y_test, y_pred)
print("Stacking F1", stacking_f1)

Stacking F1 0.8921568627450981


In [49]:
# final model evaluation

cm = confusion_matrix(y_test, y_pred)
cm

array([[71, 11],
       [11, 91]])

In [50]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.87      0.87      0.87        82
           1       0.89      0.89      0.89       102

    accuracy                           0.88       184
   macro avg       0.88      0.88      0.88       184
weighted avg       0.88      0.88      0.88       184



In [51]:
auc = roc_auc_score(y_test, y_prob)
print("ROC-AUC:", auc)

ROC-AUC: 0.9367527498804401


In [52]:
# saving the model

import joblib

joblib.dump(stack_model, "../models/stacking_classifier_model.pkl")

['../models/stacking_classifier_model.pkl']